# KadiPy Market - Module Pricing & Ingestion

Ce notebook démontre l'utilisation de `Pricing` pour l'ingestion, la normalisation et la détection d'anomalies sur les prix du marché.

---

## 1. Initialisation du Système

L'exemple utilise `Pricing(sim=True)` pour générer des prix sans réseau. 
Cela signifie : *"Ne cherche pas à te connecter à Internet, utilise le simulateur de données interne"*.

**Pourquoi ?** Cela garantit que notre logique métier est prête et dispose d'une source de données valide pour travailler, sans risquer de planter pour des problèmes de réseau ou de clé API.

In [ ]:

from kadi.market.pricing import Pricing

# 1. Initialisation avec le mode hors-ligne forcé (pour l'exemple sans clé API)
pricing = Pricing(sim=True)

print("Module Pricing initialisé avec succès !")

---
## 2. Récupération des données (`fetch`)

La méthode `fetch` est la porte d'entrée pour récupérer l'historique des marchés. En mode simulation, elle génère les prix du maïs (`maize`) à Savalou sur les 30 derniers jours (`days=30`).

**Explication de la sortie :**
La sortie est un tableau (DataFrame pandas) qui possède une structure standardisée essentielle pour KadiPy :
- `date` : Le moment du relevé.
- `price` : La valeur numérique du prix sur le marché (ici, des prix générés aléatoirement dans une fourchette réaliste par le simulateur interne).
- `unit` : Toujours formaté, ici `XOF/kg` (Francs CFA par kilo).

In [ ]:
df_prices = pricing.fetch(crop='maize', market='savalou_market', days=30)
display(df_prices.head())

---
## 3. Normalisation des Unités (`convert_unit`)

Dans le monde agricole, les prix sont rapportés de manière chaotique : en Tonnes, en sacs de 100kg, ou au kilo. 
La méthode `convert_unit` prend un prix brut, lit son unité d'origine (`unit`), et le ramène mathématiquement à une échelle standard universelle pour KadiPy (le kilogramme).

**Explication de la sortie :**
Le système reconnaît le mot "Tonne" et divise automatiquement le prix par 1000. C'est crucial : si notre IA de prévision mélangeait des prix au kilo et des prix à la tonne, elle produirait des prédictions totalement erronées.

In [ ]:
prix_tonne = 250000.0
prix_kg = pricing.convert_unit(prix_tonne, unit='XOF/Tonne', crop='maize')
print(f"{prix_tonne} XOF/Tonne = {prix_kg} XOF/kg")

---
## 4. Détection d'anomalies par Z-Score (`anomalies`)

Les collecteurs de données sur le terrain font parfois des erreurs de frappe (ajouter un 0 de trop). 
La méthode `anomalies` scanne l'historique complet en utilisant un algorithme statistique appelé **Z-Score**. Il calcule la moyenne des prix et l'écart-type (la volatilité). Si un point de donnée s'éloigne trop de la norme (généralement 3 fois l'écart-type), le système marque ce point comme suspect.

**Explication de la sortie :**
Dans la cellule ci-dessous, nous injectons volontairement un prix de 1500 XOF/kg (très élevé pour le maïs). 
Le tableau final n'affiche que cette ligne avec une nouvelle colonne `is_anomaly = True`. Le système a compris de lui-même que 1500 FCFA/kg pour du maïs est totalement incohérent vis-à-vis des autres prix. Le module d'Aide à la Décision ou de Prévision ignorera cette ligne biaisée grâce à ce tag.

In [ ]:
# Ajout d'une anomalie évidente à la fin de notre DataFrame (1500 FCFA/kg)
df_prices.loc[df_prices.index[-1], 'price'] = 1500.0

# Application de l'algorithme de détection
df_clean = pricing.anomalies(df_prices)

# Affichage uniquement des anomalies trouvées
display(df_clean[df_clean['is_anomaly'] == True])